# Model and quantization comparison
Nine GGUF variants and GPT Terra. This notebook only reads local results; it makes no API calls.

Run the README workflow first. Select one experiment below. Missing configurations remain visible in the audit. Older results can be explored by changing individual manifest entries; do not relabel FP16 or Q4_K_M as BF16 or Q4_0.

Accuracy and relevance are judge-based binary rates over available valid judgments. Failure rate uses all simulation rows. Workflow latency includes retrieval, serving, and network time; client token counts are estimates. Terra judges its own answers, which may introduce bias. Question-level bootstrap intervals do not measure variation between repeated runs.

In [62]:
from pathlib import Path
import sys
import json
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'run_open_weight_simulation.py').exists())
sys.path.insert(0, str(ROOT))
import comparative_analysis
importlib.reload(comparative_analysis)
from comparative_analysis import (manifest, load_runs, summarize, paired, bootstrap,
                                  radar_values, PERFORMANCE_RADAR, CONTEXT_RADAR)
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 40)

In [63]:
EXPERIMENT = 'study_v1'
RUNS = manifest(ROOT, EXPERIMENT)
EXPORT = ROOT / 'data/simulation/analysis' / EXPERIMENT
EXPORT.mkdir(parents=True, exist_ok=True)
# Override results/evaluation paths here if your run directories differ.
display(pd.DataFrame(RUNS))

# Save every figure as it is displayed, with deterministic sequential filenames.
_original_show = getattr(plt.show, '_original', plt.show)
_figure_counter = 0
def save_and_show(*args, **kwargs):
    global _figure_counter
    for number in plt.get_fignums():
        _figure_counter += 1
        figure = plt.figure(number)
        for suffix in ['png', 'svg']:
            figure.savefig(EXPORT / f'figure_{_figure_counter:02d}.{suffix}', bbox_inches='tight', dpi=160)
    _original_show(*args, **kwargs)
    plt.close('all')
save_and_show._original = _original_show
plt.show = save_and_show

,configuration,family,precision,run,results,evaluation
0,gemma4_12b_BF16,Gemma 4 12B,BF16,study_v1,/Users/tobialao/workspace/research/hiring-dss-...,/Users/tobialao/workspace/research/hiring-dss-...
1,gemma4_12b_Q8_0,Gemma 4 12B,Q8_0,study_v1,/Users/tobialao/workspace/research/hiring-dss-...,/Users/tobialao/workspace/research/hiring-dss-...
2,gemma4_12b_Q4_0,Gemma 4 12B,Q4_0,study_v1,/Users/tobialao/workspace/research/hiring-dss-...,/Users/tobialao/workspace/research/hiring-dss-...
3,llama31_8b_BF16,Llama 3.1 8B,BF16,study_v1,/Users/tobialao/workspace/research/hiring-dss-...,/Users/tobialao/workspace/research/hiring-dss-...
4,llama31_8b_Q8_0,Llama 3.1 8B,Q8_0,study_v1,/Users/tobialao/workspace/research/hiring-dss-...,/Users/tobialao/workspace/research/hiring-dss-...
5,llama31_8b_Q4_0,Llama 3.1 8B,Q4_0,study_v1,/Users/tobialao/workspace/research/hiring-dss-...,/Users/tobialao/workspace/research/hiring-dss-...
6,qwen35_9b_BF16,Qwen3.5 9B,BF16,study_v1,/Users/tobialao/workspace/research/hiring-dss-...,/Users/tobialao/workspace/research/hiring-dss-...
7,qwen35_9b_Q8_0,Qwen3.5 9B,Q8_0,study_v1,/Users/tobialao/workspace/research/hiring-dss-...,/Users/tobialao/workspace/research/hiring-dss-...
8,qwen35_9b_Q4_0,Qwen3.5 9B,Q4_0,study_v1,/Users/tobialao/workspace/research/hiring-dss-...,/Users/tobialao/workspace/research/hiring-dss-...
9,terra,GPT Terra,API,study_v1,/Users/tobialao/workspace/research/hiring-dss-...,/Users/tobialao/workspace/research/hiring-dss-...


In [64]:
data, audit = load_runs(RUNS)
questions = json.loads((ROOT / 'data/simulation/truthful_qa_questions.json').read_text())
cases = json.loads((ROOT / 'data/simulation/test_case_config.json').read_text())
expected_per_run = len(questions) * len(cases)
audit['expected_full_study_rows'] = expected_per_run
audit['missing_full_study_rows'] = expected_per_run - audit.simulation_rows
display(audit)
if data.empty:
    print('No selected experiment results yet. Run the README commands or update RUNS; subsequent charts will be skipped.')
else:
    display(data[['configuration', 'test_case', 'question_id', 'processing_time', 'correct', 'failed']].head())
    display(data[['processing_time', 'prompt_token_count', 'response_token_count', 'correct']].dtypes)
    display(data.groupby('configuration')[['scored', 'invalid_judge', 'failed']].agg(['sum', 'mean']))
    display(data.isna().sum().sort_values(ascending=False).head(12))
    print('Expected full-study test cases:', list(cases))
    print('Expected questions per test case:', len(questions))
    for spec in RUNS:
        config_file = Path(spec['results']) / 'RUN_CONFIG.json'
        if config_file.exists():
            print(spec['configuration'], json.loads(config_file.read_text()))

,configuration,simulation_rows,evaluation_rows,orphan_evaluations,manifest_present,expected_full_study_rows,missing_full_study_rows
0,gemma4_12b_BF16,0,0,0,False,150,150
1,gemma4_12b_Q8_0,0,0,0,False,150,150
2,gemma4_12b_Q4_0,0,0,0,False,150,150
3,llama31_8b_BF16,0,0,0,False,150,150
4,llama31_8b_Q8_0,0,0,0,False,150,150
5,llama31_8b_Q4_0,0,0,0,False,150,150
6,qwen35_9b_BF16,0,0,0,False,150,150
7,qwen35_9b_Q8_0,0,0,0,False,150,150
8,qwen35_9b_Q4_0,0,0,0,False,150,150
9,terra,0,0,0,False,150,150


No selected experiment results yet. Run the README commands or update RUNS; subsequent charts will be skipped.


## Coverage and descriptive results
All available rows contribute to descriptive tables. Inspect coverage before comparing means. Missing judgments are excluded from quality denominators rather than counted as incorrect. Judge parse errors are shown separately; failed answers with valid judgments remain in quality rates.

In [65]:
summary = summarize(data) if not data.empty else pd.DataFrame()
if not data.empty:
    display(summary)
    coverage = data.groupby(['configuration', 'test_case']).size().unstack(fill_value=0)
    fig, axes = plt.subplots(1, 2, figsize=(17, 6))
    sns.heatmap(coverage, annot=True, fmt='g', ax=axes[0], cmap='Blues')
    axes[0].set_title('Observed question coverage')
    rates = data.groupby('configuration')[['failed', 'invalid_judge', 'scored']].mean()
    rates.plot.barh(ax=axes[1])
    axes[1].set_title('Failure, invalid judge, and scoring rates')
    axes[1].set_xlim(0, 1)
    plt.tight_layout()
    plt.show()

In [66]:
if not data.empty:
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    sns.boxplot(data=data, y='configuration', x='processing_time', ax=axes[0], showfliers=False)
    axes[0].set(xlabel='Workflow latency (seconds)', title='Latency distribution (outliers hidden)')
    latency = data.groupby(['test_case', 'configuration']).processing_time.median().unstack()
    sns.heatmap(latency, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[1])
    axes[1].set_title('Median workflow latency by test case (seconds)')
    plt.tight_layout()
    plt.show()

In [67]:
if not data.empty:
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    quality = summary.set_index('configuration')[['accuracy', 'answer_relevance', 'context_relevance']]
    sns.heatmap(quality, annot=True, fmt='.1%', vmin=0, vmax=1, cmap='YlGn', ax=axes[0])
    axes[0].set_title('Quality rates on valid judgments')
    for i, (name, group) in enumerate(data.groupby('configuration', sort=False)):
        vals = group.correct.dropna()
        if vals.empty:
            continue
        mean = vals.mean()
        lo, hi = bootstrap(vals)
        axes[1].scatter(mean, i)
        if np.isfinite(lo):
            axes[1].plot([lo, hi], [i, i])
    axes[1].set_yticks(range(data.configuration.nunique()), data.configuration.unique())
    axes[1].set(xlim=(-.02, 1.02), xlabel='Accuracy', title='Accuracy with 95% question bootstrap intervals')
    plt.tight_layout()
    plt.show()

In [68]:
if not summary.empty:
    fig, ax = plt.subplots(figsize=(12, 7))
    sns.scatterplot(data=summary, x='median_latency', y='accuracy', hue='family', style='precision', s=130, ax=ax)
    for row in summary.itertuples():
        ax.annotate(row.configuration, (row.median_latency, row.accuracy), fontsize=8, xytext=(5, 5), textcoords='offset points')
    ax.set(xlabel='Median workflow latency (seconds)', ylabel='Accuracy', title='Quality and latency trade-off')
    plt.tight_layout()
    plt.show()

## Paired comparisons
Differences use identical (run, test case, question ID) keys and valid values for each metric. Positive accuracy differences favor the compared model; positive latency differences mean it was slower. Each pair may have different coverage, shown as n. These are descriptive comparisons on shared questions, without claims of statistical significance.

Quantization comparisons use each family's BF16 run as its reference. The global shared-question table below offers a stricter comparison across every configuration that has valid scores; missing configurations are explicitly listed.

In [69]:
terra_comparison = paired(data) if not data.empty else pd.DataFrame()
quantization_comparison = pd.DataFrame()
if not data.empty:
    available = set(data.configuration)
    print('Absent configurations:', sorted(set(x['configuration'] for x in RUNS) - available))
    counts = data[data.scored].groupby(['run', 'test_case', 'question_id']).configuration.nunique()
    shared = counts[counts.eq(len(available))].reset_index()[['run', 'test_case', 'question_id']]
    common_data = data.merge(shared, on=['run', 'test_case', 'question_id'])
    print('Globally shared scored questions:', len(shared), 'across', len(available), 'available configurations')
    if not common_data.empty:
        display(summarize(common_data))
    display(terra_comparison)
    parts = []
    for family, group in data[data.precision.ne('API')].groupby('family'):
        baseline = group.loc[group.precision.eq('BF16'), 'configuration']
        if not baseline.empty:
            part = paired(group, baseline.iloc[0])
            part['family'] = family
            parts.append(part)
    quantization_comparison = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    display(quantization_comparison)

In [70]:
def difference_chart(table, title):
    if table.empty:
        print(title + ': no paired observations.')
        return
    fig, axes = plt.subplots(1, 2, figsize=(17, 6))
    for ax, metric, label in zip(axes, ['correct', 'processing_time'], ['Accuracy difference', 'Mean latency difference (seconds)']):
        subset = table[table.metric.eq(metric)].reset_index(drop=True)
        for i, row in subset.iterrows():
            ax.scatter(row.difference, i)
            if np.isfinite(row.low):
                ax.plot([row.low, row.high], [i, i])
        ax.set_yticks(range(len(subset)), [f'{r.configuration} (n={r.n})' for r in subset.itertuples()])
        ax.axvline(0, color='grey', linestyle='--')
        ax.set_xlabel(label)
    fig.suptitle(title + ' — paired 95% bootstrap intervals')
    plt.tight_layout()
    plt.show()

difference_chart(terra_comparison, 'Compared with Terra')
difference_chart(quantization_comparison, 'Quantization compared with family BF16')

Compared with Terra: no paired observations.
Quantization compared with family BF16: no paired observations.


In [71]:
if not data.empty:
    fig, axes = plt.subplots(1, 2, figsize=(17, 7))
    sns.boxplot(data=data, y='configuration', x='path_length', ax=axes[0])
    axes[0].set_title('Processing-path length')
    tokens = data.groupby('configuration')[['prompt_token_count', 'response_token_count']].mean()
    tokens.plot.barh(ax=axes[1])
    axes[1].set_title('Mean client-estimated workflow tokens')
    plt.tight_layout()
    plt.show()

## Coverage carried forward from the original notebook
| Original analysis | Updated equivalent |
|---|---|
| Duration boxplots and mean-duration bars by test case | Family-panel latency distributions and mean-latency bars below; median/p95 retained |
| Overall performance radar | Performance radar below, with corrected labels and throughput |
| Context/failure radar and conditional metrics | Context/failure radar and extended summaries below |
| Throughput distribution | Output workflow throughput distribution by configuration |
| Batch quality bars and four-panel trends | Per-test-case quality bars and correctness/relevance/hallucination-proxy trends |
| Relevance vs path length | Numeric score trends and binary relevance-rate trends by configuration |
| Time vs agent calls and scatter | Workflow latency vs recorded path entries, with seconds as the unit |
| Overall, batch, and path summaries | Extended overall, per-test-case, per-configuration/path-length CSV tables |
| Correct/relevance disagreement inspection | Raw-label consistency counts; labels are never rewritten |

Repeated debug tables and commented-out experiments are consolidated into the summaries. No measured energy data existed in the original notebook.

### Restored metric definitions
All composite proxies and conditional rates require all three valid boolean judgments. Hallucination proxy = answer relevant AND (incorrect OR context irrelevant); retrieval-failure proxy = incorrect AND context irrelevant; confident-hallucination proxy = answer relevant AND retrieval-failure proxy. These names preserve the original definitions; they do not establish hallucination, model confidence, retrieval causality, or context-induced errors directly.

Conditional rates are P(incorrect | context miss), P(correct | relevant context), P(context miss | incorrect), and context sensitivity = P(correct | relevant context) − P(correct | context miss). Missing or zero denominators remain NaN. Incorrect-answer rate differs from execution failure rate.

Output workflow throughput = response tokens / positive workflow seconds, averaged over valid per-question values. It is not decoding speed. Numeric relevancy scores are preserved independently on [0,1]; in the current binary evaluator they equal their corresponding relevance rates. Missing numeric scores are not imputed. Raw correct-but-answer-irrelevant judgments are counted for review.

Path length means recorded workflow-path entries, not a verified count of model API calls.

In [72]:
overall_extended = summarize(data)
per_test_case = summarize(data, ['configuration', 'family', 'precision', 'test_case'])
per_path = summarize(data, ['configuration', 'family', 'precision', 'path_length'])
radar_table = radar_values(overall_extended) if not overall_extended.empty else pd.DataFrame()
# Equal-test-case averages preserve the original final-summary view.
# Counts remain in per_test_case; these averages weight each observed test case equally.
macro_metrics = ['accuracy', 'answer_relevance', 'context_relevance', 'hallucination_proxy',
                 'failure_given_context_miss', 'success_given_context', 'context_miss_share_among_errors',
                 'context_sensitivity', 'mean_latency', 'mean_path_length', 'output_throughput']
macro_by_test_case = (per_test_case.groupby(['configuration', 'family', 'precision'])[macro_metrics].mean().reset_index()
                     if not per_test_case.empty else pd.DataFrame())
if not data.empty:
    display(overall_extended)
    display(per_test_case)
    display(per_path)
    display(macro_by_test_case)
    display(data.groupby('configuration')[['judge_disagreement', 'hallucination_proxy', 'retrieval_failure_proxy', 'confident_hallucination_proxy']].agg(['mean', 'count']))
else:
    print('Extended analyses await simulation results.')

Extended analyses await simulation results.


### Performance and context/failure radars
Each panel compares a family's available precisions with Terra. All panels share axis order and scale. Throughput, path length, and latency are divided by the maximum configuration mean across the selected experiment, without reversing direction. Raw values accompany the radar in the extended table. An all-zero axis stays zero; unavailable values produce gaps.

↑ means higher is better; ↓ means lower is better. The mixed directions mean a larger polygon is not an overall quality score. Conditional rates use different denominators, shown in the summaries. Normalization depends on the selected configurations and must not be compared across experiments without fixing the reference maxima.

In [73]:
def family_groups(frame):
    families = list(dict.fromkeys(spec['family'] for spec in RUNS if spec['configuration'] != 'terra'))
    if not families:
        families = ['GPT Terra']
    for family in families:
        yield family, frame[frame.family.eq(family) | frame.configuration.eq('terra')]

def plot_radars(table, metric_spec, title):
    if table.empty:
        print(title + ': no results.')
        return
    panels = list(family_groups(table))
    fig, axes = plt.subplots(1, len(panels), figsize=(7 * len(panels), 8), subplot_kw={'projection': 'polar'}, squeeze=False)
    columns, labels = zip(*metric_spec)
    angles = np.linspace(0, 2 * np.pi, len(columns), endpoint=False)
    closed_angles = np.r_[angles, angles[0]]
    colors = {'BF16': '#0072B2', 'Q8_0': '#009E73', 'Q4_0': '#D55E00', 'API': '#333333'}
    for ax, (family, subset) in zip(axes.flat, panels):
        for row in subset.itertuples(index=False):
            values = np.array([getattr(row, column) for column in columns], dtype=float)
            if not np.isfinite(values).any():
                continue
            ax.plot(closed_angles, np.r_[values, values[0]], marker='o', markersize=3,
                    label=row.configuration, color=colors.get(row.precision),
                    linestyle='--' if row.configuration == 'terra' else '-')
        ax.set_theta_offset(np.pi / 2)
        ax.set_theta_direction(-1)
        ax.set_xticks(angles, labels, fontsize=9)
        ax.tick_params(axis='x', pad=18)
        ax.set_ylim(0, 1)
        ax.set_yticks([.25, .5, .75, 1])
        ax.set_title(family, pad=48)
        if ax.lines:
            ax.legend(loc='upper center', bbox_to_anchor=(.5, -.17), fontsize=9)
    fig.suptitle(title, y=.99)
    fig.subplots_adjust(top=.78, bottom=.24, left=.08, right=.92, wspace=.7)
    plt.show()

plot_radars(radar_table, PERFORMANCE_RADAR, 'Performance radar — mixed benefit/cost directions')
plot_radars(radar_table, CONTEXT_RADAR, 'Context/failure radar — conditional associations')

Performance radar — mixed benefit/cost directions: no results.
Context/failure radar — conditional associations: no results.


### Test-case latency, throughput, and quality
Family panels include Terra as a reference. Lines connect categorical test cases for readability; they do not imply a time trend. Percent axes use valid judgments; tables retain [0,1] rates and denominator counts.

In [74]:
if not data.empty:
    for family, subset in family_groups(data):
        fig, axes = plt.subplots(1, 3, figsize=(21, 6))
        sns.boxplot(data=subset, x='test_case', y='processing_time', hue='configuration', showfliers=False, ax=axes[0])
        axes[0].set(title='Latency distributions', ylabel='Workflow seconds')
        means = per_test_case[per_test_case.family.eq(family) | per_test_case.configuration.eq('terra')]
        sns.barplot(data=means, x='test_case', y='mean_latency', hue='configuration', errorbar=None, ax=axes[1])
        axes[1].set(title='Mean latency', ylabel='Workflow seconds')
        throughput = subset.dropna(subset=['output_throughput'])
        if not throughput.empty:
            sns.histplot(data=throughput, x='output_throughput', hue='configuration', bins=25, element='step', fill=False, common_bins=True, ax=axes[2])
        axes[2].set(title='Output throughput distribution', xlabel='Estimated response tokens / workflow second')
        for ax in axes[:2]:
            ax.tick_params(axis='x', rotation=30)
            ax.legend(fontsize=8)
        fig.suptitle(family)
        plt.tight_layout()
        plt.show()

    for family, subset in family_groups(per_test_case):
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        for ax, column, label in zip(axes.flat,
                ['accuracy', 'answer_relevance', 'context_relevance', 'hallucination_proxy'],
                ['Correctness', 'Answer relevance', 'Context relevance', 'Hallucination proxy']):
            for name, group in subset.groupby('configuration', sort=False):
                group = group.sort_values('test_case')
                ax.plot(group.test_case, group[column] * 100, marker='o', label=name)
            ax.set(title=label, xlabel='Test case', ylabel='Percent', ylim=(0, 100))
        handles, labels = axes[0, 0].get_legend_handles_labels()
        fig.legend(handles, labels, loc='lower center', ncol=2)
        fig.suptitle(family + ' — quality by test case')
        fig.tight_layout(rect=(0, .09, 1, .96))
        plt.show()

        fig, axes = plt.subplots(1, 3, figsize=(20, 6))
        for ax, column in zip(axes, ['accuracy', 'answer_relevance', 'context_relevance']):
            plot_data = subset.copy()
            plot_data[column] *= 100
            sns.barplot(data=plot_data, x='test_case', y=column, hue='configuration', errorbar=None, ax=ax)
            ax.set(title=column.replace('_', ' ').title(), ylabel='Percent', ylim=(0, 100))
            ax.tick_params(axis='x', rotation=30)
            if ax.legend_:
                ax.legend_.remove()
        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc='lower center', ncol=2)
        fig.suptitle(family + ' — grouped quality rates')
        fig.tight_layout(rect=(0, .13, 1, .96))
        plt.show()

### Relevance and latency versus recorded path length
These plots show associations with workflow-path entries, not causal effects of agent calls. Numeric score curves retain supplied values independently of the binary relevance curves. Tables include observation counts; small groups should be interpreted cautiously.

In [75]:
if not data.empty:
    for family, subset in family_groups(per_path):
        fig, axes = plt.subplots(1, 3, figsize=(21, 6))
        palette = dict(zip(subset.configuration.unique(), sns.color_palette('colorblind', subset.configuration.nunique())))
        for name, group in subset.groupby('configuration', sort=False):
            group = group.sort_values('path_length')
            axes[0].plot(group.path_length, group.relevancy_score, marker='o', label=name + ' answer', color=palette[name])
            axes[0].plot(group.path_length, group.context_relevancy_score, marker='s', linestyle='--', label=name + ' context', color=palette[name])
            axes[1].plot(group.path_length, group.answer_relevance, marker='o', label=name + ' answer', color=palette[name])
            axes[1].plot(group.path_length, group.context_relevance, marker='s', linestyle='--', label=name + ' context', color=palette[name])
            axes[2].plot(group.path_length, group.mean_latency, marker='o', label=name, color=palette[name])
        for ax in axes:
            ax.set_xlabel('Recorded workflow-path entries')
        axes[0].set(title='Numeric relevancy scores', ylabel='Mean score', ylim=(0, 1))
        axes[1].set(title='Binary relevance rates', ylabel='Rate', ylim=(0, 1))
        axes[2].set(title='Latency vs path length', ylabel='Mean workflow seconds')
        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc='lower center', ncol=4, fontsize=8)
        fig.suptitle(family)
        fig.tight_layout(rect=(0, .18, 1, .96))
        plt.show()

    for family, subset in family_groups(data):
        fig, ax = plt.subplots(figsize=(10, 6))
        sns.scatterplot(data=subset, x='processing_time', y='path_length', hue='configuration', alpha=.55, ax=ax)
        ax.set(title=family + ' — latency and path length', xlabel='Workflow latency (seconds)', ylabel='Recorded workflow-path entries')
        plt.tight_layout()
        plt.show()

## Export
Figures are saved automatically as PNG and SVG when displayed. The following cell exports tidy metrics and summary tables without prompts, resumes, or answer text.

In [76]:
EXPORT.mkdir(parents=True, exist_ok=True)
audit.to_csv(EXPORT / 'coverage_audit.csv', index=False)
if not data.empty:
    columns = ['configuration', 'family', 'precision', 'run', 'test_case', 'question_id', 'processing_time', 'prompt_token_count', 'response_token_count', 'path_length', 'correct', 'answer_relevant', 'context_relevant', 'failed', 'scored', 'invalid_judge', 'relevancy_score', 'context_relevancy_score', 'output_throughput', 'hallucination_proxy', 'retrieval_failure_proxy', 'confident_hallucination_proxy', 'judge_disagreement', 'incorrect_answer', 'context_miss']
    data[columns].to_csv(EXPORT / 'question_metrics.csv', index=False)
    summary.to_csv(EXPORT / 'summary.csv', index=False)
    terra_comparison.to_csv(EXPORT / 'paired_terra.csv', index=False)
    quantization_comparison.to_csv(EXPORT / 'paired_quantization.csv', index=False)
    overall_extended.to_csv(EXPORT / 'extended_overall.csv', index=False)
    per_test_case.to_csv(EXPORT / 'extended_per_test_case.csv', index=False)
    per_path.to_csv(EXPORT / 'extended_per_path.csv', index=False)
    macro_by_test_case.to_csv(EXPORT / 'macro_by_test_case.csv', index=False)
    radar_table.to_csv(EXPORT / 'radar_values.csv', index=False)
print('Tables and displayed PNG/SVG figures exported to', EXPORT)

Tables and displayed PNG/SVG figures exported to /Users/tobialao/workspace/research/hiring-dss-simulation/data/simulation/analysis/study_v1
